[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxischa/datacamp_test/blob/main/bloc2_donnees/exercices/seance4_exercices.ipynb)

# Séance 2.4 — Visualiser et conclure — étude de cas

**Exercices** · durée : 2h — 1h de technique en alternance, 1h d'étude de cas en binôme

> ⚠️ **Avant de taper quoi que ce soit :** *Fichier → Enregistrer une copie dans Drive*. Sinon votre travail sera perdu en fermant l'onglet.
>
> 📱 Sur tablette, faites d'abord les réglages de [Bien démarrer](https://github.com/maxischa/datacamp_test/blob/main/ressources/setup_tablette.md).

## Objectifs

À la fin de cette séance, vous saurez :

- choisir le bon graphique selon la question posée
- produire une courbe, un histogramme, un diagramme en barres et un nuage de points
- rendre un graphique lisible : titre, axes, unités
- repérer ce qu'un graphique cache autant que ce qu'il montre
- conclure une analyse par des recommandations chiffrées

## L'étude de cas

👥 **En binôme.** Un tient le clavier, l'autre lit l'énoncé et vérifie — puis
vous échangez à mi-parcours. C'est la façon dont on travaille réellement sur
une analyse.

Sept étapes, dans l'ordre. Chacune se termine par une **vérification** qui vous
dit immédiatement si votre résultat est bon. La dernière est le livrable : une
note de trois recommandations, chacune appuyée sur un chiffre que vous aurez
calculé vous-même.

> 💡 Vous avez tout ce qu'il faut : les quatre graphiques de la première heure,
> et tout le bloc 2 derrière vous. Aucune commande nouvelle n'est nécessaire.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Affichage adapte aux petits ecrans
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 80)

# Les donnees sont lues directement depuis le web : rien a telecharger
BASE = "https://raw.githubusercontent.com/maxischa/datacamp_test/main/bloc2_donnees/data/"

In [ ]:
def verifier(nom, condition, indice=""):
    """Affiche un retour immediat sans interrompre le notebook."""
    print("OK   -", nom) if condition else print("A REVOIR -", nom, ":", indice)

Chargement des données utilisées dans toute la feuille :

In [ ]:
ventes = pd.read_csv(BASE + "ventes.csv")
clients = pd.read_csv(BASE + "clients.csv")
produits = pd.read_csv(BASE + "produits.csv")

ventes["ca"] = ventes["qte"] * ventes["prix"]     ## le CA de chaque ligne
ventes["date"] = pd.to_datetime(ventes["date"])   ## du texte vers des dates

# Deux jointures enchainees : ventes + clients, puis + produits
complet = ventes.merge(clients, on="client_id").merge(produits, on="prod_id")
print(complet.shape)   ## 45 123 lignes : aucune perdue en chemin

### Le contexte

> **Votre mission :**
> Votre direction prépare le budget de l'an prochain. Elle vous demande une note d'une page : **où est le chiffre d'affaires, et où sont les risques ?** Les sept étapes ci-dessous vous y mènent. La dernière est le livrable.

In [ ]:
# Rien a completer ici : executez simplement la cellule
print("Donnees chargees :", complet.shape[0], "lignes")

In [ ]:
verifier("0 - donnees pretes", len(complet) == 45123, "relancez la cellule de preparation")

### Étape 1 — La saisonnalité

> **Votre mission :**
> - Calculer le CA par mois dans `ca_mois` (index = le mois en texte, ex. `"2011-10"`).
> - Tracer une **courbe**, avec titre et libellé d'axe.
> - Mettre le meilleur mois dans `mois_top`.

In [ ]:
verifier("1 - meilleur mois", mois_top == "2011-10",
         "groupby sur le mois puis sum() sur ca")

### Étape 2 — Le piège de décembre

> **Votre mission :**
> - Le graphique montre une chute en décembre. **Avant de conclure**, comptez les jours de décembre présents dans les données → `jours_dec`.
> - Puis répondez : la chute est-elle réelle ? Mettez `True` ou `False` dans `chute_reelle`.

In [ ]:
verifier("2a - jours de decembre", jours_dec == 8, "nunique() sur .dt.day")
verifier("2b - interpretation", chute_reelle is False,
         "8 jours face a des mois de 30 : les periodes ne sont pas comparables")

### Étape 3 — Les marchés

> **Votre mission :**
> - CA par pays, top 8, en **barres horizontales** triées.
> - Mettre le deuxième marché dans `marche_2`.

In [ ]:
verifier("3 - deuxieme marche", marche_2 == "Irlande",
         "nlargest trie deja : l'index 1 est le deuxieme")

### Étape 4 — La concentration client

> **Votre mission :**
> - Calculer le CA par client, puis la part des **10 premiers** dans le CA total → `part_top10` (en %, arrondi à 1 décimale).
> - Compter les clients irlandais → `nb_irl`.
> - Ces deux chiffres sont le cœur de votre note.

In [ ]:
verifier("4a - part des 10 premiers", part_top10 == 37.5,
         "divisez la somme des 10 premiers par le total ca_cli.sum()")
verifier("4b - clients irlandais", nb_irl == 2, "nunique() sur client_id")

### Étape 5 — Le top produits, et ce qu'il révèle

> **Votre mission :**
> - Afficher les 5 produits qui génèrent le plus de CA → `top_prod`.
> - **Regardez les noms attentivement.** Deux d'entre eux ne sont pas des produits.
> - Mettre leurs deux libellés dans la liste `faux_produits`.

In [ ]:
verifier("5 - faux produits reperes", sorted(faux_produits) == ["Manual", "Postage"],
         "un classement produits ne devrait pas contenir de frais de port")

### Étape 6 — Le classement corrigé

> **Votre mission :**
> - Refaire le top 5 en excluant `Postage` et `Manual` → `top_reel`.
> - Calculer la part de ces deux lignes dans le CA total → `part_faux` (en %, arrondi à 1 décimale).

In [ ]:
verifier("6a - vrai produit leader", top_reel.index[0] == "Regency Cakestand 3 Tier",
         "filtrez avec 'not in @faux_produits' avant de classer")
verifier("6b - part des faux produits", part_faux == 6.2,
         "utilisez 'in @faux_produits' pour isoler ces deux lignes")

### Étape 7 — Le livrable

> **Votre mission :**
> - Rédigez votre note dans la cellule markdown ci-dessous, en remplaçant les points de suspension.
> - **Trois recommandations, chacune appuyée sur un chiffre que vous avez calculé.**
> - Un constat n'est pas une recommandation : « l'Irlande fait 22,7 % du CA » est un constat ; « il faut sécuriser ces deux contrats » est une recommandation.
> - 👥 Chaque binôme présentera 3 minutes.

In [ ]:
verifier("7 - tous les chiffres disponibles",
         all(v is not None for v in [mois_top, marche_2, part_top10, part_faux]),
         "reprenez les etapes 1 a 6 avant de rediger")
print()
print("A vous : ajoutez une cellule de texte et redigez vos 3 recommandations.")